In [1]:
# Typical steps
# 1. Application parses policy documents and extracts relevant information to create a knowledge base.

# This data will be stored in vector database and will be used as context for question answering.
content_corpus = [
        "Employees are compensated on a bi-weekly basis through direct deposit.",       
        "Employees must submit a leave request for approval.", 
        "Company internet must be used for work-related tasks only.",
        "Company internet is a broadband internet.",
        "Employees can take an hour break.",
        "Interact with each employee with Respect"
]

content_corpus[3]

'Company internet is a broadband internet.'

In [2]:
# Goal: Retrieve relevant documents based on user query (a.k.a. Context Retrieval)

%pip install -q sentence-transformers


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_vectors = model.encode(content_corpus)

/Users/amiteshsinha/Training/2026_10_Gen_Agentic_AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1115 22:03:48.738000 65316 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8897.77it/s]


In [4]:
doc_vectors

array([[-0.01192887,  0.01236365,  0.03582984, ...,  0.01726841,
         0.07261264, -0.0317501 ],
       [ 0.03315501,  0.0485338 ,  0.04736276, ...,  0.10182013,
         0.09159281,  0.00358369],
       [-0.07135906, -0.03066473,  0.0318377 , ..., -0.04109798,
         0.06524781, -0.00688532],
       [-0.00383739, -0.02336754,  0.02958674, ..., -0.04415295,
         0.12559088, -0.03139855],
       [-0.01790447,  0.01495855,  0.08163837, ..., -0.03217228,
        -0.00513644,  0.05279532],
       [-0.00240887,  0.03361145, -0.06162642, ...,  0.04830884,
         0.03707634, -0.01683051]], shape=(6, 384), dtype=float32)

In [5]:
# Step 3: User Query and Semantic Matching

query = "what's the internet policy?"
model = SentenceTransformer("all-MiniLM-L6-v2")
query_vec = model.encode([query])[0]
query_vec

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9106.90it/s]


array([-8.68199859e-03, -2.51570921e-02,  1.28027443e-02, -6.54579848e-02,
        6.82432530e-03,  3.36576141e-02,  5.55899553e-02, -6.23868667e-02,
       -3.04177944e-02, -4.82558273e-03,  4.20290232e-02,  9.65184048e-02,
       -1.64934248e-02, -5.40239960e-02,  3.31156552e-02,  7.06644845e-04,
        1.08587230e-02, -6.68311939e-02, -7.00457171e-02, -6.10482059e-02,
        5.77069670e-02, -2.66837385e-02,  1.74021199e-02,  1.17126703e-02,
       -1.52451470e-02,  1.06889093e-02, -8.20444990e-03,  3.93415093e-02,
       -8.79153162e-02,  2.42121182e-02, -6.55871630e-03, -7.95376003e-02,
       -8.73170284e-05, -2.12384146e-02, -8.67849961e-02, -5.71142212e-02,
       -8.35197419e-02, -3.80301513e-02, -6.08348884e-02,  9.35147866e-04,
        6.14595637e-02, -9.44368988e-02, -1.20505141e-02,  6.37602285e-02,
        7.73343667e-02,  2.63111927e-02,  3.64377871e-02,  3.56474034e-02,
       -3.49588208e-02, -3.32529796e-03,  4.55280021e-02,  1.36845615e-02,
        2.34139189e-02,  

In [6]:
similarities = model.similarity(query_vec, doc_vectors)
print(type(similarities))

# Ensure it's a 1D numpy array
import numpy as np
similarities = np.asarray(similarities).squeeze()
similarities

<class 'torch.Tensor'>


array([0.03589284, 0.07394685, 0.4535125 , 0.533936  , 0.04768127,
       0.03751658], dtype=float32)

In [7]:
# Now get top 3 indices based on similarity scores
top_3_indices = np.argsort(similarities)[::-1][:3]
print(top_3_indices)
top_scores = similarities[top_3_indices]
top_scores

[3 2 1]


array([0.533936  , 0.4535125 , 0.07394685], dtype=float32)

In [8]:
top_scores

array([0.533936  , 0.4535125 , 0.07394685], dtype=float32)

In [37]:
documents = content_corpus
top_docs = [documents[i] for i in top_3_indices]


print (top_docs)
context = ", ".join(top_docs)
context

['Company internet is a broadband internet.', 'Company internet must be used for work-related tasks only.', 'Employees must submit a leave request for approval.']


'Company internet is a broadband internet., Company internet must be used for work-related tasks only., Employees must submit a leave request for approval.'

In [38]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

my_client = OpenAI(api_key=my_api_key)
# my_client

def ask_question_open_ai(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create
    # context = getweather(Zipcode="10001", Country="US")
    print(f"Context: {context}")
    print(f"User Question: {query}")    
    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        # messages=[
        #     {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
        #     {"role": "user", "content": prompt}
        # ]
        messages=[
            {"role": "system", "content": '''
            #  You are an assistant who answers only based on the given context. Use every context information even if
            #  it is not related to the user question.
            You are an assistant who answers only based on the given context. 
            Invent facts if necessary and include them in the answer. Be creative. 
            Look at most poular answers for this question.
            
            
             '''},
            {"role": "user", "content": f"Context: {context}\n\n User Question: {query}"} 
        ]

    )
    return llm_response.choices[0].message.content  


In [39]:
response = ask_question_open_ai(query)

print(f"User query: {query}")
print(f"Context: {context}")

print(f"\n\nOpen AI Response: {response}")

Context: Company internet is a broadband internet., Company internet must be used for work-related tasks only., Employees must submit a leave request for approval.
User Question: Who developed Python, and in what year?
User query: Who developed Python, and in what year?
Context: Company internet is a broadband internet., Company internet must be used for work-related tasks only., Employees must submit a leave request for approval.


Open AI Response: Guido van Rossum developed Python, and the first public release was in 1991. (He started the project in the late 1980s at the Centrum Wiskunde & Informatica in the Netherlands.)

Note: Our company policy says the internet is for work-related tasks only. If you’re doing Python development on company resources, please submit a leave request for approval.


In [ ]:
#Reference Data
expected_answer = "The company's internet must be used for work-related tasks only."    
actual_response = response

# Call ChatCompltions API to compare expected vs actual
# LLM AS A JUDGE vs Traditional string comparison
comparison_prompt = f"""
You are an expert evaluator. Compare the actual response to the expected answer and determine if they match in meaning.
Expected Answer: {expected_answer}
Actual Response: {actual_response}
Do they match in meaning? Answer with 'Yes' or 'No' and provide a brief explanation. Score them from 0 to 1 in terms of how closely they match in meaning.
"""
comparison_response = my_client.chat.completions.create(
    model="gpt-6-luna",     
    messages=[
        {"role": "system", "content": "You are an expert evaluator."},
        {"role": "user", "content": comparison_prompt}
    ]
)   
print(f"\nExpected Answer: {comparison_response.choices[0].message.content}")

### Context Recall

In [36]:
# Context Recall
# User query is in query variable
ground_truth = '''The company provides broadband internet,
 Internet use must be for work-related tasks only.'''
#Actual context is in context variable
print(f"Actual context: {context}")
print(f"Ground truth: {ground_truth}")
print(f"Query: {query}")


Actual context: Python was created by Guido van Rossum.
Ground truth: The company provides broadband internet,
 Internet use must be for work-related tasks only.
Query: Who developed Python, and in what year?


In [ ]:


comparison_prompt = f"""
You are an expert evaluator. Compute context recall - here are the facts
- Query: {query}
- Ground Truth: {ground_truth}
- Context retrieved was this : {context}
Compute context recall. Score from 0 to 1. Provide the score in the format of "Context Recall: <score>".

"""
comparison_response = my_client.chat.completions.create(
    model="gpt-6-luna",     
    messages=[
        {"role": "system", "content": "You are an expert evaluator."},
        {"role": "user", "content": comparison_prompt}
    ]
)   
print(f"\nExpected Answer: {comparison_response.choices[0].message.content}")

In [ ]:

# Call ChatCompltions API to compare expected vs actual
# LLM AS A JUDGE vs Traditional string comparison
comparison_prompt = f"""
The user asked this: “Who developed Python?” - We gave this context: “Python was created by Guido van Rossum in 1991.”
The answer was “Python was created by Guido van Rossum.”

Did the LLM generate a faithful answer ot the context. Score from 0 to 1.

"""
comparison_response = my_client.chat.completions.create(
    model="gpt-6-luna",     
    messages=[
        {"role": "system", "content": "You are an expert evaluator."},
        {"role": "user", "content": comparison_prompt}
    ]
)   
print(f"\nExpected Answer: {comparison_response.choices[0].message.content}")


Expected Answer: 1 — The answer faithfully states that Guido van Rossum created Python. It omits the year, but that isn’t needed to answer the question.


In [ ]:
print (f"\nExpected Answer: {expected_answer}")
print (f"\nActual Response: {actual_response}")

In [ ]:
# Context Recall
query = "Who developed Python, and in what year?"
ground_truth = "Guido van Rossum, 1991"
context = "Python was created by Guido van Rossum."

comparison_prompt = f"""
You are an expert evaluator. Compute context recall - here are the facts
- Query: {query}
- Ground Truth: {ground_truth}
- Context retrieved was this : {context}
Compute context recall. Score from 0 to 1. Provide the score in the format of "Context Recall: <score>".

"""
comparison_response = my_client.chat.completions.create(
    model="gpt-6-luna",     
    messages=[
        {"role": "system", "content": "You are an expert evaluator."},
        {"role": "user", "content": comparison_prompt}
    ]
)   
print(f"\nExpected Answer: {comparison_response.choices[0].message.content}")


Expected Answer: Context Recall: 0.5
